# 03 — ERA5 variables (Bogotá)

Downloads hourly ERA5 single-level fields over a small box around Bogotá
(lat 4.5, lon -74.05, per Casallas et al. 2022) via `cdsapi`, then aggregates to daily
mean/min/max. Only 3 variables, deliberately: boundary-layer height (no ground
instrument measures this), surface solar radiation downwards, and total precipitation —
none of which appear in any local `Data/COLOMBIA` file. Wind, temperature, RH and
pressure are **not** pulled from ERA5; Bogotá's SISAIRE station gauges (notebook 01/02)
are the better source for those and are already in the cleaned dataset.

The download step queues on the Copernicus Climate Data Store and can take a while —
run this notebook non-interactively (e.g. via `jupyter nbconvert --execute`) rather than
cell-by-cell.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import pandas as pd
from common.external_data import download_era5_hourly, load_era5_daily, BOGOTA_ERA5_AREA

OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"
RAW_ERA5_DIR = PROJECT_ROOT / "Data" / "external" / "era5"

network_daily = pd.read_parquet(OUT_DIR / "bogota_network_daily_clean.parquet")
start_year = network_daily['date'].dt.year.min()
end_year = network_daily['date'].dt.year.max()
print("Requesting ERA5 for", start_year, "-", end_year, "over area", BOGOTA_ERA5_AREA)


Requesting ERA5 for 2014 - 2025 over area [4.9, -74.35, 4.35, -73.85]


In [2]:
# A single multi-year, multi-variable request exceeds the CDS per-request cost limit
# ("cost limits exceeded"), so this downloads one year at a time into separate files.
era5_yearly_paths = download_era5_hourly(start_year, end_year, RAW_ERA5_DIR)
era5_yearly_paths


[download_era5_hourly] blh 2014 already downloaded, skipping
[download_era5_hourly] blh 2015 already downloaded, skipping
[download_era5_hourly] blh 2016 already downloaded, skipping
[download_era5_hourly] blh 2017 already downloaded, skipping
[download_era5_hourly] blh 2018 already downloaded, skipping
[download_era5_hourly] blh 2019 already downloaded, skipping
[download_era5_hourly] blh 2020 already downloaded, skipping
[download_era5_hourly] blh 2021 already downloaded, skipping
[download_era5_hourly] blh 2022 already downloaded, skipping
[download_era5_hourly] blh 2023 already downloaded, skipping
[download_era5_hourly] blh 2024 already downloaded, skipping
[download_era5_hourly] blh 2025 already downloaded, skipping
[download_era5_hourly] ssrd 2014 already downloaded, skipping
[download_era5_hourly] ssrd 2015 already downloaded, skipping
[download_era5_hourly] ssrd 2016 already downloaded, skipping
[download_era5_hourly] ssrd 2017 already downloaded, skipping
[download_era5_hourl

[PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2014.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2015.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2016.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2017.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2018.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2019.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2020.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2021.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2022.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2023.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2024.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_blh_2025.nc'),
 PosixPath('/path/to/project/Data/external/era5/bogota_era5_ssrd_2014.nc'),
 PosixPath('/path/to/pro

In [3]:
era5_daily = load_era5_daily(era5_yearly_paths)
print(era5_daily.shape)
era5_daily.head()


(4384, 7)


,date,blh_mean,blh_min,blh_max,blh_morning_mean,radiation_mean,precip_total_mm
0,2013-12-31,101.584023,69.267044,142.505692,NaN,0.000000,0.401616
1,2014-01-01,548.785400,48.970657,1754.936035,149.582504,223.997025,0.928283
2,2014-01-02,510.346710,38.262131,1527.504395,207.655838,228.505371,0.901699
3,2014-01-03,422.381500,67.865570,1193.630859,96.910316,201.284439,1.125455
4,2014-01-04,431.068512,67.462906,1247.625244,135.824570,145.772964,0.967145


In [4]:
out_path = OUT_DIR / "bogota_era5_daily.parquet"
era5_daily.to_parquet(out_path, index=False)
print("Saved:", out_path, era5_daily.shape)


Saved: /path/to/project/Data/processed/colombia/bogota/bogota_era5_daily.parquet (4384, 7)


## Lower Tropospheric Stability (LTS)

Added per user request after reviewing v3 results — the discovered equation leaned
heavily on a raw day-of-year term, which doesn't explain *why* PM2.5 has a seasonal
cycle. LTS (`theta_700 - theta_surface`) is a standard atmospheric-stability diagnostic:
a strong inversion (high LTS) traps pollution near the surface. The group's own papers
(`Lagrangian_manuscript.pdf`) cite a similar Estimated Inversion Strength diagnostic as
"essential in explaining pollution variations" for NWSA cities.

This needs a **different CDS collection** (`reanalysis-era5-pressure-levels`, not
`reanalysis-era5-single-levels`) for temperature at 700 hPa, plus ERA5's own 2m
temperature and surface pressure (pulled fresh here, not reused from station data,
so the potential-temperature calculation is internally self-consistent within one
reanalysis model).


In [5]:
from common.external_data import download_era5_pressure_level_hourly, compute_lts_daily

# t2m/sp (single-level) via the existing helper; t700 (pressure-level) via the new one.
lts_singlelevel_paths = download_era5_hourly(
    start_year, end_year, RAW_ERA5_DIR, variables=["2m_temperature", "surface_pressure"]
)
t700_paths = download_era5_pressure_level_hourly(start_year, end_year, RAW_ERA5_DIR, pressure_level="700")


[download_era5_hourly] t2m_lts 2014 already downloaded, skipping
[download_era5_hourly] t2m_lts 2015 already downloaded, skipping
[download_era5_hourly] t2m_lts 2016 already downloaded, skipping
[download_era5_hourly] t2m_lts 2017 already downloaded, skipping
[download_era5_hourly] t2m_lts 2018 already downloaded, skipping
[download_era5_hourly] t2m_lts 2019 already downloaded, skipping
[download_era5_hourly] t2m_lts 2020 already downloaded, skipping
[download_era5_hourly] t2m_lts 2021 already downloaded, skipping
[download_era5_hourly] t2m_lts 2022 already downloaded, skipping
[download_era5_hourly] t2m_lts 2023 already downloaded, skipping
[download_era5_hourly] t2m_lts 2024 already downloaded, skipping
[download_era5_hourly] t2m_lts 2025 already downloaded, skipping
[download_era5_hourly] sp_lts 2014 already downloaded, skipping
[download_era5_hourly] sp_lts 2015 already downloaded, skipping
[download_era5_hourly] sp_lts 2016 already downloaded, skipping
[download_era5_hourly] sp_lt

[download_era5_pressure_level_hourly] t700 2014 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2015 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2016 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2017 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2018 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2019 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2020 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2021 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2022 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2023 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2024 already downloaded, skipping
[download_era5_pressure_level_hourly] t700 2025 already downloaded, skipping


In [6]:
t2m_paths = [p for p in lts_singlelevel_paths if "t2m_lts" in p.name]
sp_paths = [p for p in lts_singlelevel_paths if "sp_lts" in p.name]

lts_daily = compute_lts_daily(t2m_paths, sp_paths, t700_paths)
print(lts_daily.shape)
print(lts_daily['lts'].describe())
lts_daily.head()


(4384, 3)
count    4384.000000
mean        0.516313
std         0.671361
min        -2.098900
25%         0.080840
50%         0.563623
75%         0.988566
max         2.835873
Name: lts, dtype: float64


,date,lts,lts_morning
0,2013-12-31,1.794183,NaN
1,2014-01-01,0.426231,2.908752
2,2014-01-02,0.576689,2.961782
3,2014-01-03,0.195368,2.293365
4,2014-01-04,0.477356,2.506653


In [7]:
lts_out_path = OUT_DIR / "bogota_lts_daily.parquet"
lts_daily.to_parquet(lts_out_path, index=False)
print("Saved:", lts_out_path, lts_daily.shape)


Saved: /path/to/project/Data/processed/colombia/bogota/bogota_lts_daily.parquet (4384, 3)


## v6: dispersion/mixing physics beyond LTS (user request)

After the v5 result (R²=0.493 physical, R²=0.41 PySR), the user wants to push toward
R²>=0.7 while staying physical (no `pm25_lag1`). New single-level pulls:
`total_column_water_vapour` (TCWV, integrated moisture -- distinct from surface RH),
`surface_sensible_heat_flux` (thermal driver of daytime mixing), and 10m u/v wind
components (**not** to replace station wind_speed as the daily-mean predictor -- pulled
specifically to compute a within-day turbulence/TKE-like proxy from hourly variance,
which needs ERA5's sub-daily resolution that station daily-means can't provide). Plus
one pressure-level pull: vertical velocity at 700 hPa (`w700`) -- a genuinely different
physical axis from LTS (synoptic-scale subsidence/instability vs. temperature-based
inversion strength).

(Mobility data considered and set aside: Google/Apple COVID mobility reports are
permanently discontinued, covering only Feb 2020-Oct 2022 at best -- ~21% of our
2014-2025 record, worse coverage than the fire-hotspot gap we already work around.)


In [8]:
from common.external_data import load_vertical_velocity_daily

v6_singlelevel_paths = download_era5_hourly(
    start_year, end_year, RAW_ERA5_DIR,
    variables=["total_column_water_vapour", "surface_sensible_heat_flux",
               "10m_u_component_of_wind", "10m_v_component_of_wind"],
)
w700_paths = download_era5_pressure_level_hourly(
    start_year, end_year, RAW_ERA5_DIR, pressure_level="700", variable="vertical_velocity"
)


[download_era5_hourly] tcwv 2014 already downloaded, skipping
[download_era5_hourly] tcwv 2015 already downloaded, skipping
[download_era5_hourly] tcwv 2016 already downloaded, skipping
[download_era5_hourly] tcwv 2017 already downloaded, skipping
[download_era5_hourly] tcwv 2018 already downloaded, skipping
[download_era5_hourly] tcwv 2019 already downloaded, skipping
[download_era5_hourly] tcwv 2020 already downloaded, skipping
[download_era5_hourly] tcwv 2021 already downloaded, skipping
[download_era5_hourly] tcwv 2022 already downloaded, skipping
[download_era5_hourly] tcwv 2023 already downloaded, skipping
[download_era5_hourly] tcwv 2024 already downloaded, skipping
[download_era5_hourly] tcwv 2025 already downloaded, skipping
[download_era5_hourly] sshf 2014 already downloaded, skipping
[download_era5_hourly] sshf 2015 already downloaded, skipping
[download_era5_hourly] sshf 2016 already downloaded, skipping
[download_era5_hourly] sshf 2017 already downloaded, skipping
[downloa

[download_era5_pressure_level_hourly] w700 2014 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2015 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2016 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2017 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2018 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2019 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2020 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2021 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2022 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2023 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2024 already downloaded, skipping
[download_era5_pressure_level_hourly] w700 2025 already downloaded, skipping


In [9]:
# Recompute era5_daily including the original vars + the new v6 single-level ones
# (load_era5_daily merges whatever's present; original blh/ssrd/tp files still exist).
all_singlelevel_paths = era5_yearly_paths + v6_singlelevel_paths
era5_daily = load_era5_daily(all_singlelevel_paths)
print(era5_daily.shape)
print(era5_daily[['tcwv', 'sensible_heat_flux', 'tke_proxy']].describe())

out_path = OUT_DIR / "bogota_era5_daily.parquet"
era5_daily.to_parquet(out_path, index=False)
print("Saved:", out_path, era5_daily.shape)


(4384, 12)
              tcwv  sensible_heat_flux    tke_proxy
count  4384.000000         4384.000000  4384.000000
mean     19.303198          -49.376934     0.200073
std       2.927058           17.780123     0.095345
min       6.072098         -125.371620     0.035234
25%      17.681342          -58.959667     0.133031
50%      19.671509          -47.239357     0.178869
75%      21.350555          -37.377275     0.248100
max      27.029661           -3.565833     0.835985
Saved: /path/to/project/Data/processed/colombia/bogota/bogota_era5_daily.parquet (4384, 12)


In [10]:
w700_daily = load_vertical_velocity_daily(w700_paths)
print(w700_daily.shape)
print(w700_daily['w700'].describe())

w700_out_path = OUT_DIR / "bogota_w700_daily.parquet"
w700_daily.to_parquet(w700_out_path, index=False)
print("Saved:", w700_out_path, w700_daily.shape)


(4384, 2)
count    4384.000000
mean        0.006173
std         0.134341
min        -0.457930
25%        -0.088851
50%        -0.003046
75%         0.091392
max         0.551823
Name: w700, dtype: float64
Saved: /path/to/project/Data/processed/colombia/bogota/bogota_w700_daily.parquet (4384, 2)
